In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns

# 读取数据（需要将Excel文件放在同级目录）
df = pd.read_excel("销售订单汇总.xlsx", sheet_name="Sheet1")

rows, columns = df.shape

df = df.loc[6:rows, ['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 6']]

# 数据预处理
df.columns = ['Round', 'Day', 'Material', 'Description', 'Qty']
df['Qty'] = df['Qty'].replace(',', '', regex=True).astype(float)  # 处理千分位分隔符
df['Day'] = df['Day'].astype(int)
df['Round'] = df['Round'].astype(int)

# 获取所有产品列表（按Material排序）
products = sorted(df['Material'].unique().tolist())

# 配置绘图参数
sns.set_theme()  # 设置Seaborn的绘图风格
colors = plt.cm.tab20.colors  # 使用预定义颜色

# 按轮次绘制图表
for round_num in sorted(df['Round'].unique(), reverse=True):
    round_data = df[df['Round'] == round_num]

    if round_num == 5:  # 判断轮次是否为第 5 轮，如果是则跳过
        continue
    
    if round_data.empty:
        continue
        
    max_qty = 1
    for product in products:
        product_data = round_data[round_data['Material'] == product]
        if not product_data.empty:
            days = range(1, 21)
            qty = product_data.set_index('Day').reindex(days)['Qty'].fillna(0)
            current_max = qty.max()
            if current_max > max_qty:
                max_qty = current_max

    # 创建画布
    fig, axs = plt.subplots(2, 6, figsize=(30, 10))

    # 设置中文字体为黑体
    plt.rcParams['font.sans-serif'] = ['SimHei']
    # 解决负号显示问题
    plt.rcParams['axes.unicode_minus'] = False
    
    fig.suptitle(f'Round {round_num} 产品销量趋势', fontsize=20, y=1.02)

    # 绘制每个产品的子图
    for idx, product in enumerate(products):
        ax = axs[idx//6, idx % 6]
        product_data = round_data[round_data['Material'] == product]

        if not product_data.empty:
            # 确保日期连续
            days = range(1, 21)
            qty = product_data.set_index('Day').reindex(days)['Qty'].fillna(0)

            ax.plot(days, qty,
                    marker='o',
                    color=colors[idx],
                    linewidth=2,
                    markersize=6)

            # 设置坐标轴
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
            ax.set_xticks(range(1, 21, 2))
            ax.set_ylim(bottom=0, top=max_qty)

            # 添加标题和标签
            title = f"{product}\n{product_data['Description'].iloc[0]}"
            ax.set_title(title, fontsize=10, pad=8)
            ax.set_xlabel('Day', fontsize=8)
            ax.set_ylabel('Qty', fontsize=8)

        else:
            ax.axis('off')  # 无数据的子图留白

    # 调整布局
    plt.tight_layout()
    
    plt.subplots_adjust(hspace=0.4, wspace=0.3)

    # 保存图片
    plt.savefig(f'Round_{round_num}_Sales_Trend.png',
                bbox_inches='tight',
                dpi=300)
    plt.close()